# Free Sliders Interactive Notebook — Text to Video (CogVideoX)

This notebook mirrors `T2I_interactive.ipynb` and `T2A_interactive.ipynb`, but targets the CogVideoX-2b backbone. It runs on macOS Apple Silicon via the `mps` device or on CUDA / CPU automatically.

**Methods Available:**
- **FreeSliders**: noise mixing during diffusion (uses `CogVideoXDPMScheduler`).
- **TE**: linear interpolation of text embeddings.

**Note:** CogVideoX-2b is a ~10GB model. The first run will download weights from HuggingFace. On Apple Silicon expect a few minutes per scale at 480x720x24f.

## 1. Import Required Libraries

In [1]:
import os
import sys
import warnings
import importlib
from datetime import datetime

import torch
from IPython.display import Video, display

warnings.filterwarnings("ignore", category=FutureWarning)

from methods.FreeSliders import FreeSliders
from methods.TE import TE
from utils.config_utils import load_config_from_yaml
from utils.prompt_utils import PromptSettings, load_prompts_from_yaml
from utils.utils import get_device

device = get_device()
print(f"Using device: {device}")

Using device: cuda


## 2. Load Configuration and Prompts

Choose between **static** prompts (camera/scene composition concepts) and **dynamic** prompts (motion-related concepts).

In [2]:
config_file = "configs/config.yaml"
config = load_config_from_yaml(config_file)

video_cfg = getattr(config, "video", None)
static_prompts_file = (
    video_cfg.static_prompts_file if video_cfg is not None else "configs/static_prompts.yaml"
)
dynamic_prompts_file = (
    video_cfg.dynamic_prompts_file if video_cfg is not None else "configs/dynamic_prompts.yaml"
)

static_prompts = load_prompts_from_yaml(static_prompts_file, [])
dynamic_prompts = load_prompts_from_yaml(dynamic_prompts_file, [])
print(f"Loaded {len(static_prompts)} static concepts from {static_prompts_file}")
for p in static_prompts:
    print(f"  - {p.tag}: {p.neutral!r}")
print(f"\nLoaded {len(dynamic_prompts)} dynamic concepts from {dynamic_prompts_file}")
for p in dynamic_prompts:
    print(f"  - {p.tag}: {p.neutral!r}")

class NotebookLogger:
    def log_hparams(self, params): pass
    def add_tags(self, tag): pass
    def stop(self): pass
    def log(self, name, data, step=None): pass
    def log_audio(self, name, path): pass
    def log_video(self, name, path): print(f"{name}: saved video at {path}")
    def log_fig(self, name, fig): pass
    def __getitem__(self, key): return self

run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
logger = NotebookLogger()

Loaded 12 static concepts from configs/static_prompts.yaml
  - mountain_hiking: 'A mountaintop from afar with a few people hiking down it.'
  - car_type: 'A car cruising down the street.'
  - car_color: 'A car cruising down the street.'
  - mountain_type: 'A large mountain in the distance.'
  - cup: 'A pitcher pouring clear water into a cup on the table.'
  - tree: 'A tall tree in a windy park.'
  - cat_fat: 'A cat sitting on a windowsill licking its paws.'
  - cat_color: 'A cat sitting on a windowsill licking its paws.'
  - lighthouse_day: 'Waves splashing on a lighthouse in the sea.'
  - lighthouse_water: 'A lighthouse in the sea, surrounded by water.'
  - dog_color: 'A dog playing in the park.'
  - butterfly_colors: 'A butterfly landing on a leaf.'

Loaded 10 dynamic concepts from configs/dynamic_prompts.yaml
  - sailboat: 'A white sailboat in the ocean.'
  - river_waves: 'A river flowing through the valley.'
  - butterfly_wings: 'A butterfly on a leaf.'
  - train: 'A train moving a

## 3. Load CogVideoX-2b

First-time downloads of the model weights can take several minutes. On MPS the model runs in float32; on CUDA it uses bfloat16.

In [3]:
print("Loading CogVideoX-2b...")
module_name = "base_models.BaseModelCogVideoX"
BaseModelClass = importlib.import_module(module_name).BaseModel
base_model = BaseModelClass(config, run_id)
print(f"Model loaded on {base_model.device} (dtype={base_model.weight_dtype}).")

Loading CogVideoX-2b...


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded on cuda (dtype=torch.bfloat16).


## 4. Choose Concepts and Encode Prompts

Pick `static` or `dynamic` prompts and which concept indices to run.

In [4]:
prompt_kind = "static"  # or "dynamic"
selected_indices = [0]

source_prompts = static_prompts if prompt_kind == "static" else dynamic_prompts
selected_prompts = [source_prompts[i] for i in selected_indices]
for p in selected_prompts:
    print(f"\nTag: {p.tag}\n  neutral : {p.neutral}\n  positive: {p.positive}\n  negative: {p.negative}")

all_prompt_embeddings = base_model.encode_prompts(selected_prompts)
print(f"\nEncoded {len(all_prompt_embeddings)} prompt pairs")
base_model.delete_and_flush()


Tag: mountain_hiking
  neutral : A mountaintop from afar with a few people hiking down it.
  positive: A mountaintop from afar with a lot of people hiking down it, very busy.
  negative: A mountaintop from afar with a no people hiking down it, empty.

Encoded 1 prompt pairs


## 5. Generation Parameters

In [5]:
selected_method = "FreeSliders"  # or "TE"

if selected_method == "FreeSliders":
    scales = [-3.0, 0.0, 3.0]
    guidance_scale_variation = 1.0
elif selected_method == "TE":
    scales = [-1.5, 0.0, 1.5]
    guidance_scale_variation = 6.0

num_images = 1
timestep_intervention = 15  # of 50 total CogVideoX steps
guidance_scale_neutral = 6.0
seed = 42

class Args:
    def __init__(self, scales, tag, method, kind):
        self.num_images = num_images
        self.t = timestep_intervention
        self.scales = scales
        self.guidance = guidance_scale_neutral
        self.guidance2 = guidance_scale_variation
        self.BaseModel = "T2V"
        self.concept_tag = tag
        self.method = method
        self.static = (kind == "static")
        self.dynamic = (kind == "dynamic")

print(f"Method: {selected_method} | scales: {scales} | t={timestep_intervention} | guidance2={guidance_scale_variation}")

Method: FreeSliders | scales: [-3.0, 0.0, 3.0] | t=15 | guidance2=1.0


## 6. Run the Method

Generated `.mp4` files land under `generated_videos/<method>/<tag>/<seed>/scale_<scale>.mp4`. Latents are cached under `T2V_sd_latents/...` (FreeSliders) or `T2V_latents/EmbedsInter/...` (TE).

In [6]:
if selected_method == "FreeSliders":
    method_model = FreeSliders(base_model)
elif selected_method == "TE":
    method_model = TE(base_model)
else:
    raise ValueError(f"Unknown method: {selected_method}")

all_results = {}
for idx, prompt in zip(selected_indices, selected_prompts):
    args = Args(scales, prompt.tag, selected_method, prompt_kind)
    text_embeddings = {prompt_kind: [all_prompt_embeddings[selected_indices.index(idx)]]}
    prompts = {prompt_kind: [prompt]}
    results = method_model(
        args=args,
        text_embeddings=text_embeddings,
        prompts=prompts,
        seed=seed,
        logger=logger,
    )
    all_results[prompt.tag] = results
    print(f"Finished {prompt.tag}")


Computing FreeSliders for - Tag: mountain_hiking
Neutral Prompt: A mountaintop from afar with a few people hiking down it.
Positive Prompt: A mountaintop from afar with a lot of people hiking down it, very busy.
Negative Prompt: A mountaintop from afar with a no people hiking down it, empty.
Exporting videos...
Computing metrics...


/home/nvidia/Free_Sliders/evaluation/viclip_utils.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


  0%|          | 0/3 [00:00<?, ?it/s]

Skipping video metrics — evaluator setup failed: Failed to initialize viclip_text_alignment metrics: [Errno 2] No such file or directory: '/home/nvidia/Free_Sliders/evaluation/metric_models/ViClip-InternVid-10M-FLT.pth'
Finished mountain_hiking


## 7. Playback

Find and display the generated `.mp4` files for each tag/scale.

In [7]:
import glob

for tag in all_results.keys():
    print(f"\n=== {tag} ===")
    pattern = f"generated_videos/{selected_method}/{tag}/**/scale_*.mp4"
    paths = sorted(glob.glob(pattern, recursive=True))
    if not paths:
        print("  (no videos found)")
        continue
    for path in paths:
        scale = os.path.basename(path).replace("scale_", "").replace(".mp4", "")
        print(f"scale={scale}  ({path})")
        display(Video(path, embed=True))


=== mountain_hiking ===
scale=-3.0  (generated_videos/FreeSliders/mountain_hiking/913/scale_-3.0.mp4)


scale=0.0  (generated_videos/FreeSliders/mountain_hiking/913/scale_0.0.mp4)


scale=3.0  (generated_videos/FreeSliders/mountain_hiking/913/scale_3.0.mp4)
